<a href="https://colab.research.google.com/github/joaomerjam/ECON3916-33674-Statistical-Machine-Learning/blob/main/Lab%2012/Lab12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

# Step 1: Ingestion from external source
url = 'https://raw.githubusercontent.com/joaomerjam/ECON3916-33674-Statistical-Machine-Learning/refs/heads/main/Lab%2012/Data/Zillow_ZHVI_2026_Micro.csv'
df = pd.read_csv(url)

In [2]:
df.head()

,Home_Value,Square_Footage,Property_Age,Distance_to_Transit,School_District_Rating
0,329705.74,1941.0,5.5,6.45,Excellent
1,183343.63,1364.3,35.2,2.15,Average
2,354551.73,2386.9,52.4,0.75,Good
3,325773.17,2192.1,50.2,5.25,Excellent
4,359743.12,3069.8,66.5,12.69,Excellent


In [5]:
# Step 2: Defining the formula
# Utilizing the R-style patsy formula interface allows for elegant, readable model specification
formula ='Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit + School_District_Rating'


In [7]:
# Step 3: Fitting the model and printing the summary
model = smf.ols(formula=formula, data=df)
results = model.fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     542.5
Date:                Mon, 16 Mar 2026   Prob (F-statistic):          2.81e-309
Time:                        19:57:46   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.416e+04
Df Residuals:                     993   BIC:                         2.419e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [13]:
# Step 4: Generating predictions
# We extract the predicted values vector to transition from explanation to prediction
y_pred = results.predict()

In [11]:
y_pred

array([312288.58686592, 223813.44090966, 331610.31628442, 307402.42680623,
       392714.85172172, 340728.94426174, 254692.12823547, 202854.09435084,
       369429.13502941, 327783.49088826, 309246.00435669, 258360.3807938 ,
       139094.88308995, 379873.30953205, 347185.24220689, 329568.68169377,
       397984.32960147, 197734.71132874, 459576.88043278, 297587.13946194,
       331543.77981689, 372358.44833234,  87395.94396635, 450100.61209461,
       288424.70313068, 223533.6898128 , 275036.20646947, 532307.17687308,
       322033.78931033, 221131.17401575, 281053.99679786, 338150.27960139,
       278456.57490773, 304783.54769867, 407542.98164548, 381021.3426314 ,
       312961.62526799, 324477.19487721, 304111.11626134, 287717.94577641,
       291954.27932085, 349343.90768117, 431554.52856937, 246866.51544979,
       377335.99535948, 424153.06505514, 261216.06175632, 343146.76187735,
       307464.77557608, 324692.07656789, 269334.16793959, 234943.37180521,
       136408.88582822, 3

In [14]:
# Step 5: Calculate RMSE between the actuals and the predictions
model_rmse = rmse(df.Home_Value, y_pred)
print(f"\nThe Predictive RMSE is: ${model_rmse:,.2f}")


The Predictive RMSE is: $42,316.69


## Gemini Attempt

# Task
Prepare the data for plotting by extracting residuals from the model results, calculating their standard deviation, identifying outliers based on a 2-standard deviation threshold, and then create an interactive residual plot using `plotly.express` to visualize the relationship between predicted values and residuals, highlighting outliers in crimson. Follow this with an explanation of how to interpret the residual plot for diagnostic purposes and a summary of the residual forensics dashboard.

## Prepare Data for Residual Plot

### Subtask:
Extract the residuals from the `statsmodels` results object, calculate the standard deviation of the residuals, and identify outliers based on a 2-standard deviation threshold. Combine predicted values, actual values, residuals, and an outlier flag into a new DataFrame suitable for plotting.


**Reasoning**:
I will extract the residuals, calculate their standard deviation, identify outliers based on a 2-standard deviation threshold, and then compile these along with actual and predicted values into a new DataFrame for plotting, as instructed by the subtask.



In [15]:
residuals = results.resid
std_residuals = residuals.std()
outliers = (np.abs(residuals) > 2 * std_residuals)

plot_df = pd.DataFrame({
    'Actual_Home_Value': df['Home_Value'].reset_index(drop=True),
    'Predicted_Home_Value': y_pred,
    'Residuals': residuals,
    'Outlier': outliers
})

print("Created plot_df DataFrame with actuals, predictions, residuals, and outlier flags.")
plot_df.head()

Created plot_df DataFrame with actuals, predictions, residuals, and outlier flags.


,Actual_Home_Value,Predicted_Home_Value,Residuals,Outlier
0,329705.74,312288.586866,17417.153134,False
1,183343.63,223813.440910,-40469.810910,False
2,354551.73,331610.316284,22941.413716,False
3,325773.17,307402.426806,18370.743194,False
4,359743.12,392714.851722,-32971.731722,False


**Reasoning**:
The subtask requires creating an interactive residual plot with `plotly.express`, visualizing the relationship between predicted values and residuals, and highlighting outliers. This code block will achieve that by using the `plot_df` DataFrame created in the previous step and applying conditional coloring for outliers.



In [16]:
import plotly.express as px

# Step 6: Create an interactive residual plot
fig = px.scatter(
    plot_df,
    x='Predicted_Home_Value',
    y='Residuals',
    color='Outlier',
    color_discrete_map={True: 'crimson', False: 'darkblue'},
    title='Residual Plot of Home Value Predictions',
    labels={
        'Predicted_Home_Value': 'Predicted Home Value',
        'Residuals': 'Residuals (Actual - Predicted)',
        'Outlier': 'Outlier ( > 2 Std Dev)'
    },
    hover_data={'Actual_Home_Value': True, 'Predicted_Home_Value': True, 'Residuals': True, 'Outlier': True}
)

fig.update_traces(marker=dict(size=5, opacity=0.7))
fig.update_layout(
    hovermode='closest',
    xaxis_title='Predicted Home Value',
    yaxis_title='Residuals',
    legend_title='Outlier Status'
)

fig.show()

print("Generated interactive residual plot.")

Generated interactive residual plot.


### Interpreting the Residual Plot for Diagnostic Purposes

The residual plot is a crucial diagnostic tool for assessing the validity of a regression model. Here's how to interpret it:

1.  **Randomness of Residuals**: Ideally, residuals should be randomly scattered around the horizontal line at zero. This indicates that the model is capturing the underlying patterns in the data and that there are no systematic errors.
    *   **What to look for**: A random cloud of points with no discernible pattern (e.g., no U-shape, no funnel shape).
    *   **What it means if random**: The assumption of linearity and homoscedasticity (constant variance of errors) might be met. Also, it suggests that relevant predictors have been included, and the chosen functional form is appropriate.

2.  **Homoscedasticity (Constant Variance)**: The spread of the residuals should be roughly constant across all predicted values. If the spread changes (e.g., widens or narrows as predicted values increase), it indicates heteroscedasticity.
    *   **What to look for**: The vertical spread of the points should be consistent across the entire range of predicted values.
    *   **What it means if heteroscedastic**: The variance of the errors is not constant, which violates an assumption of ordinary least squares (OLS) regression. This can lead to inefficient coefficient estimates and incorrect standard errors, affecting the reliability of hypothesis tests and confidence intervals. Transformations of the dependent variable or weighted least squares might be necessary.

3.  **No Clear Patterns**: The absence of any systematic pattern (e.g., curvilinear patterns, increasing/decreasing trends) suggests that the linear model is appropriate.
    *   **What to look for**: No clear curves (like a parabola or a sinusoidal wave) or trends in the residuals.
    *   **What it means if patterns exist**: A pattern suggests that the model is missing important non-linear relationships or that important predictors have been omitted. This might require adding polynomial terms, interaction terms, or considering a different model (e.g., a non-linear regression model).

4.  **Outliers and Influential Points**: Points far from the main cloud of residuals, especially those colored crimson in our plot (indicating residuals > 2 standard deviations away from the mean), are outliers.
    *   **What to look for**: Individual points that lie far from the bulk of the residuals, either vertically (large residual) or horizontally (extreme predicted value combined with a large residual).
    *   **What it means**: Outliers can disproportionately influence the regression coefficients, leading to a biased model. They might represent data entry errors, unusual observations, or suggest that the model does not adequately describe a subset of the data. Further investigation of these points is warranted. Depending on their nature, they might need to be removed, transformed, or analyzed separately.

### Summary of the Residual Forensics Dashboard

The residual plot, as generated, serves as a mini-dashboard for residual forensics:

*   **Predicted vs. Residuals**: The primary visualization directly displays the core assumption of random, independently distributed errors.
*   **Outlier Identification**: The explicit coloring of points that are more than 2 standard deviations away from the mean residual immediately highlights potential problematic data points. This allows for quick identification of observations that might be unduly influencing the model or that warrant further investigation.
*   **Interactive Features**: The interactivity (hovering for detailed information) allows users to quickly inspect the actual value, predicted value, and residual for any specific point, which is crucial for understanding the nature of outliers or patterns. This helps in pinpointing specific data rows that may need attention.
*   **Visual Assessment of Assumptions**: It provides a visual check for key OLS assumptions like linearity and homoscedasticity, which are critical for the reliability of the model's inferences.

In essence, this residual plot helps to visually confirm if the assumptions of the linear regression model are met and to identify any systematic issues or unusual data points that need to be addressed to improve the model's accuracy and reliability.

## Final Task

### Subtask:
Summarize the created residual forensics dashboard and its utility for model diagnostics.


## Summary:

### Q&A
*   **How should the residual plot be interpreted for diagnostic purposes?**
    The residual plot should ideally show residuals randomly scattered around the horizontal zero line, indicating that the model captures underlying patterns without systematic errors and that assumptions of linearity and homoscedasticity are likely met. Key diagnostic indicators include:
    *   **Randomness of Residuals**: A random cloud of points without discernible patterns (e.g., U-shape, funnel shape) suggests appropriate linearity and homoscedasticity.
    *   **Homoscedasticity (Constant Variance)**: The vertical spread of points should be consistent across all predicted values; varying spread indicates heteroscedasticity.
    *   **Absence of Clear Patterns**: No curvilinear patterns or trends suggest the linear model is appropriate; their presence implies missing non-linear relationships or omitted predictors.
    *   **Outliers and Influential Points**: Points far from the main cluster (especially those beyond 2 standard deviations) are outliers that may disproportionately influence the model and require further investigation.
*   **What is the utility of the created residual forensics dashboard?**
    The interactive residual plot serves as a mini-dashboard for residual forensics by:
    *   **Directly visualizing Predicted vs. Residuals**: Allowing visual assessment of the core assumption of random, independently distributed errors.
    *   **Identifying outliers**: Explicitly highlighting points beyond 2 standard deviations, enabling quick identification of potentially problematic data points.
    *   **Providing interactive features**: Allowing users to hover for detailed information (actual value, predicted value, residual) for any point, which is crucial for investigating outliers or patterns.
    *   **Enabling visual assessment of assumptions**: Offering a visual check for key OLS assumptions like linearity and homoscedasticity, which are critical for model reliability.

### Data Analysis Key Findings
*   Residuals were successfully extracted from the model results, and their standard deviation was calculated to identify outliers.
*   Outliers were defined as residuals exceeding 2 standard deviations from the mean of the residuals.
*   A `plot_df` DataFrame was created, integrating `Actual_Home_Value`, `Predicted_Home_Value`, `Residuals`, and a boolean `Outlier` flag for plotting purposes.
*   An interactive scatter plot was generated using `plotly.express`, visualizing `Predicted_Home_Value` against `Residuals`.
*   Outliers identified in the `plot_df` were distinctly highlighted in 'crimson' on the plot, while non-outliers were shown in 'darkblue', enhancing visual identification.

### Insights or Next Steps
*   Further investigate the data points identified as outliers (highlighted in crimson) to understand their nature, such as potential data entry errors, unusual observations, or specific characteristics not captured by the current model.
*   Based on the visual interpretation of the plot, if patterns (e.g., funnel shapes, curves) are observed, consider refining the model by addressing issues like heteroscedasticity (e.g., data transformations, weighted least squares) or non-linearity (e.g., adding polynomial terms, exploring non-linear models).
